# Lab 14 Challenge: worked solution

The challenge has two tasks on the lab's index: a gift-card question answered by a query engine filtered to one file,
with no agent; and a second hop on ticket 10, asked of the retriever-tool agent and of the query-engine agent. This
notebook runs on its own. It repeats what it needs from the lab (the config cell, the 36 `Document`s and their nodes,
the two models, the usage log, the index, the two tools and the two agents with the plain prompt), then solves both
tasks.

⚠️ Like the lab, it runs on the lab's own environment: choose `14_LlamaIndex_RAG/.venv` as the kernel.

## Setup: the pieces from the lab

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


The 36 files as `Document`s (the file name in the LLM view, `kind` hidden from both models), one node per file, and
the two models from `models.env` as the defaults.

In [2]:
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

CORPUS = ROOT / "data" / "corpus"


def make_document(path):
    return Document(
        text=path.read_text(encoding="utf-8"),
        id_=path.name,
        metadata={"file_name": path.name, "kind": path.name.split("_")[0]},
        excluded_embed_metadata_keys=["file_name", "kind"],
        excluded_llm_metadata_keys=["kind"],
    )


documents = [make_document(path) for path in sorted(CORPUS.glob("*.md"))]
nodes = SentenceSplitter().get_nodes_from_documents(documents)

llm = OpenAI(model=MODEL, reasoning_effort=EFFORT, max_tokens=300)
Settings.llm = llm
Settings.embed_model = OpenAIEmbedding(model=settings["EMBEDDING_MODEL"])
print(len(documents), "documents ->", len(nodes), "nodes | LLM:", Settings.llm.model,
      "| embedding:", Settings.embed_model.model_name)

36 documents -> 36 nodes | LLM: gpt-6-luna | embedding: text-embedding-3-small


The usage log from Part C: an instrumentation handler that records every model call's tokens as the API reported
them. Run it once.

In [3]:
from llama_index.core.instrumentation import get_dispatcher
from llama_index.core.instrumentation.event_handlers import BaseEventHandler
from llama_index.core.instrumentation.events.llm import LLMChatEndEvent

CALLS = []  # one entry per model call: the messages sent and the tokens the API reported


class UsageLog(BaseEventHandler):
    def handle(self, event, **kwargs):
        if isinstance(event, LLMChatEndEvent):
            usage = event.response.raw.usage
            CALLS.append({"messages": event.messages, "input": usage.prompt_tokens, "output": usage.completion_tokens})


get_dispatcher().add_event_handler(UsageLog())

The index, the two tools and the two agents of Parts F and G, with the plain prompt and the same description.

In [4]:
from llama_index.core.agent.workflow import FunctionAgent, ToolCallResult
from llama_index.core.tools import QueryEngineTool, RetrieverTool

index = VectorStoreIndex(nodes)

DESCRIPTION = (
    "Search the shop policy and customer ticket files. "
    "Argument input must be a concise keyword phrase (e.g. 'gift card expiry', 'student discount', 'Puja ticket')."
)
PROMPT = "You answer questions about the shop's policies and support tickets. Use the search tool to find the facts."

qe_tool = QueryEngineTool.from_defaults(query_engine=index.as_query_engine(), name="search", description=DESCRIPTION)
rt_tool = RetrieverTool.from_defaults(
    retriever=index.as_retriever(similarity_top_k=2), name="search", description=DESCRIPTION
)
qe_agent = FunctionAgent(tools=[qe_tool], llm=llm, system_prompt=PROMPT, streaming=False)
rt_agent = FunctionAgent(tools=[rt_tool], llm=llm, system_prompt=PROMPT, streaming=False)
print("index:", len(index.vector_store.data.embedding_dict), "vectors | agents ready")

index: 36 vectors | agents ready


---

## Task 1: A gift-card question with no agent

Our code knows the topic, so the search can be limited to one file: a `MetadataFilter` on `file_name`, the key the
documents carry for exactly this kind of use. The query engine is otherwise the default one (top 2, one model call).

In [5]:
from llama_index.core.vector_stores import MetadataFilter, MetadataFilters

GIFT_QUESTION = "Do gift cards expire, and can a customer exchange one for cash?"
gift_cards_only = MetadataFilters(filters=[MetadataFilter(key="file_name", value="policy_gift_cards.md")])
gift_engine = index.as_query_engine(filters=gift_cards_only)

start = len(CALLS)
gift_response = gift_engine.query(GIFT_QUESTION)
print(gift_response.response)
print()
for item in gift_response.source_nodes:
    print(f"source: {item.score:.4f}  {item.node.metadata['file_name']}")
print("model calls:", len(CALLS) - start)

Gift cards do not expire. They cannot be exchanged for cash unless required by law.

source: 0.6912  policy_gift_cards.md
model calls: 1


One model call and one source, `policy_gift_cards.md` (0.6912). The answer: *"Gift cards do not expire. They cannot be exchanged for cash unless required by law."*

The check in code. Each line asks one question: are all the sources the gift-card file; does the answer say the
cards do not expire ("do not", "don't" or "never" with "expire"); does it limit cash to where the law requires it
("cash" and "law" both appear).

In [6]:
answer = gift_response.response.lower()
checks = {
    "every source is policy_gift_cards.md": all(
        item.node.metadata["file_name"] == "policy_gift_cards.md" for item in gift_response.source_nodes
    ),
    "says the cards do not expire": "expire" in answer and any(w in answer for w in ["do not", "don't", "never"]),
    "cash only where the law requires it": "cash" in answer and "law" in answer,
}
for name, ok in checks.items():
    print("✅" if ok else "❌", name)

✅ every source is policy_gift_cards.md
✅ says the cards do not expire
✅ cash only where the law requires it


All three checks pass. They establish that the answer came from the one file we chose and names both points; a keyword check cannot tell a reworded rule from a wrong one, so the answer above is still worth reading. The filter left one node to search, so the default top 2 returned only one: a filter can make
`similarity_top_k` larger than the number of nodes that match, and that is not an error.

**Why no agent:** the question's topic was known before any search, so a filtered engine answers in one call. An
agent would spend at least two (one to choose a query, one to answer) to rediscover what our code already knew.

---

## Task 2: A second hop on another ticket

Ticket 10 on disk, before any run:

In [7]:
print((CORPUS / "ticket_10_chile_student_discount.md").read_text(encoding="utf-8"))
print((CORPUS / "policy_student_discount.md").read_text(encoding="utf-8"))

# Ticket 10 — Chile, student discount after invoice

Date: 2011-04-03
Customer: billing country Chile

Customer sent a student email after checkout and asked for 10 percent back. Policy: student discount is applied at checkout only.

Status: closed. No credit.

# Student discount

A verified student email (ending in .edu or the local equivalent) gets 10 percent off physical items. Digital albums are not discounted.

The discount is applied at checkout. We do not add it after the invoice exists.

The discount cannot be stacked with gift-card promotions.



The rep's question needs two facts:

1. **Can we add the discount now?** No: it is applied at checkout only. The ticket says so, and the policy says *"We do
   not add it after the invoice exists."*
2. **Would it cover a digital album?** No: *"Digital albums are not discounted."* Only `policy_student_discount.md`
   says this; the ticket's line *"Policy: student discount is applied at checkout only"* names the policy without
   quoting it.

On paper, then, the agent has to follow the ticket's pointer to the policy, as with ticket 06; the runs show whether
it needs to. `run_agent` is Part F's function,
here without the fact checks (they belong to ticket 06's question); it records the queries, the files each search
found and the answer, and the calls from the log. `POLICY_FILE` is the file that holds fact 2.

In [8]:
DISCOUNT_QUESTION = (
    "Ticket 10's customer is asking again. Can we add the student discount to her invoice now, "
    "and would it cover a digital album?"
)
POLICY_FILE = "policy_student_discount.md"


async def run_agent(agent, question):
    start = len(CALLS)
    handler = agent.run(user_msg=question, max_iterations=10)
    searches = []
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            raw = event.tool_output.raw_output
            found = raw.source_nodes if hasattr(raw, "source_nodes") else raw
            searches.append({
                "query": event.tool_kwargs["input"],
                "files": [item.node.metadata["file_name"] for item in found],
                "read": event.tool_output.content,
            })
    result = await handler
    return {"answer": result.response.content, "searches": searches, "calls": len(CALLS) - start}


async def ask_three_times(agent, show_read):
    for i in range(1, 4):
        run = await run_agent(agent, DISCOUNT_QUESTION)
        found = any(POLICY_FILE in search["files"] for search in run["searches"])
        print(f"--- run {i}: policy file found: {found} | {len(run['searches'])} searches, {run['calls']} model calls")
        for search in run["searches"]:
            print("  query: ", search["query"], "->", search["files"])
            if show_read:
                print("  read:  ", search["read"])
        print("  answer:", run["answer"])
        print()

First the retriever-tool agent. It reads the files themselves, so we print only the queries and the files each one
found.

In [9]:
await ask_three_times(rt_agent, show_read=False)

--- run 1: policy file found: True | 1 searches, 2 model calls
  query:  Ticket 10 student discount invoice digital album -> ['policy_student_discount.md', 'ticket_10_chile_student_discount.md']
  answer: No. Ticket 10’s invoice already exists, and the student discount can only be applied at checkout—not added afterward. Even when applied, it covers physical items only; digital albums are excluded.



--- run 2: policy file found: True | 1 searches, 2 model calls
  query:  Ticket 10 student discount invoice digital album -> ['policy_student_discount.md', 'ticket_10_chile_student_discount.md']
  answer: No. Ticket 10’s invoice already exists, and the student discount can only be applied at checkout—not added afterward. Even at checkout, it would apply only to physical items; digital albums are excluded.



--- run 3: policy file found: True | 2 searches, 2 model calls
  query:  Ticket 10 customer student discount invoice -> ['ticket_10_chile_student_discount.md', 'policy_student_discount.md']
  query:  student discount digital album eligible products invoice -> ['policy_student_discount.md', 'ticket_10_chile_student_discount.md']
  answer: No. Ticket 10 is closed with no credit, and the student discount can’t be added after the invoice exists—it must be applied at checkout. It also applies only to physical items, not digital albums.



The policy file was found in 3 of 3 runs, and every answer says no twice: not after the invoice exists, and not for digital albums. Now look at the queries. In runs 1 and 2 the **first** query already returned `policy_student_discount.md` beside the ticket, and the agent answered after one search. Run 3 made two searches in the same turn, and its first also returned both files.

Now the query-engine agent on the same question. Its tool returns sentences, so we print what it read as well.

In [10]:
await ask_three_times(qe_agent, show_read=True)

--- run 1: policy file found: True | 2 searches, 4 model calls
  query:  Ticket 10 student discount invoice -> ['ticket_10_chile_student_discount.md', 'policy_student_discount.md']
  read:   The 10% student discount applies only at checkout and can’t be added after an invoice is issued. Ticket 10 was closed with no credit.
  query:  student discount digital album policy -> ['policy_student_discount.md', 'policy_digital_downloads.md']
  read:   Student discounts do not apply to digital albums. A verified student email gets 10% off physical items at checkout, but the discount can’t be added after an invoice is issued or combined with gift-card promotions.
  answer: No. The 10% student discount can only be applied at checkout, so it can’t be added after the invoice has been issued. It also doesn’t apply to digital albums—only eligible physical items—and Ticket 10 was closed with no credit.



--- run 2: policy file found: True | 2 searches, 4 model calls
  query:  student discount eligible digital album add to invoice policy -> ['policy_student_discount.md', 'ticket_10_chile_student_discount.md']
  read:   Digital albums are not eligible for the student discount. The 10% discount applies only to physical items and must be applied at checkout; it cannot be added after an invoice has been issued.
  query:  Ticket 10 student discount invoice digital album -> ['policy_student_discount.md', 'ticket_10_chile_student_discount.md']
  read:   Ticket 10 is closed with no credit. The student discount must be applied at checkout and isn’t added after an invoice exists. Digital albums aren’t eligible for the discount.
  answer: No. Ticket 10 is closed with no credit, and the student discount can’t be added after an invoice has been issued—it must be applied at checkout. It also doesn’t cover digital albums; it applies only to eligible physical items.



--- run 3: policy file found: True | 2 searches, 4 model calls
  query:  student discount digital album policy -> ['policy_student_discount.md', 'policy_digital_downloads.md']
  read:   Verified students receive 10% off physical items only. Digital albums are not discounted, and the student discount cannot be added after an invoice is created or combined with a gift-card promotion.
  query:  Ticket 10 customer student discount invoice -> ['ticket_10_chile_student_discount.md', 'policy_student_discount.md']
  read:   The discount could not be added after the invoice was issued. The ticket was closed with no credit.
  answer: No. Ticket 10’s invoice has already been issued, and the student discount can’t be added afterward. Also, the 10% student discount applies only to physical items, not digital albums.



The query-engine agent also found the policy in 3 of 3 runs, with both facts in every answer, at 4 model calls a run against the retriever tool's 2. Its sentences kept both rules this time: each search returned the policy itself, and the rules are short sentences that match the queries' words.

**Why this question was easy for both agents, and ticket 06's was not.** The rep's question here says *student discount*, the policy's own words, so the first search finds the policy without following the ticket's pointer. Ticket 06's question says *replacement*, *warped* and *refund*, never *damaged media*: only the ticket's line led to the rule, and the query engine's sentences lost that line. A question needs a second hop when its words do not reach the second file. Test an agent's tool on questions like that, not only on ones the first search answers.